In [2]:
## FVG Confluence Analysis - RTH Session Direction
## Tracks FPFVG in 9:30-10am and 10-11am windows
## Measures how the RTH session (9:30am open vs 4:15pm close) resolves

import pandas as pd
import numpy as np
from collections import defaultdict

file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"

print("=" * 100)
print("FVG CONFLUENCE ANALYSIS - RTH SESSION DIRECTION")
print("=" * 100)

# ============================================================
# DATA LOADING AND RESAMPLING
# ============================================================

print("\n[1/4] Loading data...")
df = pd.read_csv(file_path)
print(f"      Loaded {len(df):,} rows")

df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.sort_index()

start_date = pd.to_datetime('2013-01-01')
end_date   = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]
print(f"      Filtered to {len(df):,} rows ({start_date.date()} to {end_date.date()})")

print("\n[2/4] Resampling to 5-minute bars...")
df_5min = df.resample('5min').agg({
    'Open':  'first',
    'High':  'max',
    'Low':   'min',
    'Close': 'last'
}).dropna()
print(f"      Resampled to {len(df_5min):,} bars")
df = df_5min

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def detect_fpfvg(bar_data):
    """
    Detect the First Presented Fair Value Gap in a slice of bars.
    BISI (Bullish): candle_3 low > candle_1 high
    SIBI (Bearish): candle_3 high < candle_1 low
    Returns 'BISI', 'SIBI', or 'None'
    """
    if len(bar_data) < 3:
        return 'None'

    for i in range(1, len(bar_data) - 1):
        c1 = bar_data.iloc[i - 1]
        c3 = bar_data.iloc[i + 1]

        if c3['Low'] > c1['High']:
            return 'BISI'
        if c3['High'] < c1['Low']:
            return 'SIBI'

    return 'None'


def safe_pct(num, denom):
    return (num / denom * 100) if denom > 0 else 0


# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

print("\n[3/4] Analysing trading days...")

# Tracking structure:
# combo_stats[window1_fvg][window2_fvg] = {'total', 'bullish', 'bearish', 'neutral'}
# window1_fvg and window2_fvg each: 'BISI', 'SIBI', 'None'

FVG_VALUES = ['BISI', 'SIBI', 'None']

combo_stats = {}
for v1 in FVG_VALUES:
    combo_stats[v1] = {}
    for v2 in FVG_VALUES:
        combo_stats[v1][v2] = {'total': 0, 'bullish': 0, 'bearish': 0, 'neutral': 0}

# Also track each window independently
window1_stats = {v: {'total': 0, 'bullish': 0, 'bearish': 0, 'neutral': 0} for v in FVG_VALUES}
window2_stats = {v: {'total': 0, 'bullish': 0, 'bearish': 0, 'neutral': 0} for v in FVG_VALUES}

# Get unique calendar dates that have RTH data
all_dates = df.index.normalize().unique()

days_processed = 0
days_skipped   = 0

for date in all_dates:
    date_str = date.strftime('%Y-%m-%d')

    # --- Define key timestamps ---
    rth_open_time   = pd.Timestamp(f"{date_str} 09:30:00")
    window1_end     = pd.Timestamp(f"{date_str} 10:00:00")
    window2_end     = pd.Timestamp(f"{date_str} 11:00:00")
    rth_close_time  = pd.Timestamp(f"{date_str} 16:15:00")  # 4:15pm

    # --- Get RTH open bar ---
    rth_open_bar = df.loc[rth_open_time:window1_end - pd.Timedelta(minutes=1)]
    if rth_open_bar.empty:
        days_skipped += 1
        continue

    rth_open_price = rth_open_bar['Open'].iloc[0]

    # --- Get RTH close bar ---
    rth_close_bar = df.loc[rth_close_time - pd.Timedelta(minutes=5):rth_close_time]
    if rth_close_bar.empty:
        # Fall back: last bar before 4:15
        fallback = df.loc[rth_open_time:rth_close_time]
        if fallback.empty:
            days_skipped += 1
            continue
        rth_close_price = fallback['Close'].iloc[-1]
    else:
        rth_close_price = rth_close_bar['Close'].iloc[-1]

    # --- RTH session direction ---
    if rth_close_price > rth_open_price:
        session_dir = 'bullish'
    elif rth_close_price < rth_open_price:
        session_dir = 'bearish'
    else:
        session_dir = 'neutral'

    # --- Window 1: 9:30am - 10:00am FPFVG ---
    w1_data = df.loc[rth_open_time:window1_end - pd.Timedelta(minutes=1)]
    fvg1 = detect_fpfvg(w1_data)

    # --- Window 2: 10:00am - 11:00am FPFVG ---
    w2_data = df.loc[window1_end:window2_end - pd.Timedelta(minutes=1)]
    fvg2 = detect_fpfvg(w2_data)

    # --- Update stats ---
    window1_stats[fvg1]['total']      += 1
    window1_stats[fvg1][session_dir]  += 1

    window2_stats[fvg2]['total']      += 1
    window2_stats[fvg2][session_dir]  += 1

    combo_stats[fvg1][fvg2]['total']     += 1
    combo_stats[fvg1][fvg2][session_dir] += 1

    days_processed += 1

print(f"      Days processed: {days_processed:,} | Skipped: {days_skipped:,}")

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n[4/4] Generating report...")

def print_outcome(stats, indent="  "):
    t = stats['total']
    if t == 0:
        print(f"{indent}No data")
        return
    print(f"{indent}Sample : {t}")
    print(f"{indent}Bullish: {stats['bullish']:4d}  ({safe_pct(stats['bullish'], t):5.1f}%)")
    print(f"{indent}Bearish: {stats['bearish']:4d}  ({safe_pct(stats['bearish'], t):5.1f}%)")
    print(f"{indent}Neutral: {stats['neutral']:4d}  ({safe_pct(stats['neutral'], t):5.1f}%)")


print("\n" + "=" * 100)
print("SECTION 1: WINDOW 1 (9:30–10:00am) FPFVG  →  RTH SESSION (9:30am open vs 4:15pm close)")
print("=" * 100)

for fvg in FVG_VALUES:
    label = {'BISI': 'BULLISH FVG (BISI)', 'SIBI': 'BEARISH FVG (SIBI)', 'None': 'NO FVG'}[fvg]
    print(f"\n  9:30–10am FPFVG = {label}")
    print(f"  {'-' * 50}")
    print_outcome(window1_stats[fvg])


print("\n" + "=" * 100)
print("SECTION 2: WINDOW 2 (10:00–11:00am) FPFVG  →  RTH SESSION")
print("=" * 100)

for fvg in FVG_VALUES:
    label = {'BISI': 'BULLISH FVG (BISI)', 'SIBI': 'BEARISH FVG (SIBI)', 'None': 'NO FVG'}[fvg]
    print(f"\n  10am–11am FPFVG = {label}")
    print(f"  {'-' * 50}")
    print_outcome(window2_stats[fvg])


print("\n" + "=" * 100)
print("SECTION 3: CONFLUENCE — WINDOW 1 + WINDOW 2 FPFVG  →  RTH SESSION")
print("=" * 100)
print("\nAll 9 combinations of Window 1 × Window 2 FVG signals:\n")

for v1 in FVG_VALUES:
    for v2 in FVG_VALUES:
        label1 = {'BISI': 'BISI (Bull)', 'SIBI': 'SIBI (Bear)', 'None': 'No FVG'}[v1]
        label2 = {'BISI': 'BISI (Bull)', 'SIBI': 'SIBI (Bear)', 'None': 'No FVG'}[v2]
        print(f"  9:30–10am = {label1:<14}  |  10am–11am = {label2:<14}")
        print(f"  {'-' * 60}")
        print_outcome(combo_stats[v1][v2])
        print()


# ============================================================
# SECTION 4: SUMMARY TABLE
# ============================================================

print("=" * 100)
print("SECTION 4: SUMMARY TABLE")
print("=" * 100)
print(f"\n{'Window 1 (9:30-10)':<20} {'Window 2 (10-11)':<20} {'N':>6} {'Bull%':>8} {'Bear%':>8} {'Neutral%':>10}")
print("-" * 76)

for v1 in FVG_VALUES:
    for v2 in FVG_VALUES:
        label1 = {'BISI': 'BISI', 'SIBI': 'SIBI', 'None': 'No FVG'}[v1]
        label2 = {'BISI': 'BISI', 'SIBI': 'SIBI', 'None': 'No FVG'}[v2]
        s = combo_stats[v1][v2]
        t = s['total']
        print(f"{label1:<20} {label2:<20} {t:>6} "
              f"{safe_pct(s['bullish'], t):>7.1f}% "
              f"{safe_pct(s['bearish'], t):>7.1f}% "
              f"{safe_pct(s['neutral'], t):>9.1f}%")


# ============================================================
# SECTION 5: WINDOW 1 (9:30-10am) FPFVG - ISOLATED PREDICTIVE TABLE
# ============================================================

print("\n" + "=" * 100)
print("SECTION 5: WINDOW 1 (9:30–10:00am) FPFVG  →  RTH SESSION — ISOLATED TABLE")
print("=" * 100)
print("\nHow predictive is the first FVG of the day (9:30–10am) for the full RTH session?")
print(f"\n{'9:30-10am FVG':<16} {'N':>6} {'Bull%':>8} {'Bear%':>8} {'Neutral%':>10}  {'Edge'}") 
print("-" * 65)

for fvg in FVG_VALUES:
    s = window1_stats[fvg]
    t = s['total']
    label = {'BISI': 'BISI (Bullish)', 'SIBI': 'SIBI (Bearish)', 'None': 'No FVG'}[fvg]
    bull_pct = safe_pct(s['bullish'], t)
    bear_pct = safe_pct(s['bearish'], t)
    neut_pct = safe_pct(s['neutral'], t)

    # Edge: for BISI show bull advantage, for SIBI show bear advantage, for None show dominant direction
    if fvg == 'BISI':
        edge = f"Bull edge: +{bull_pct - bear_pct:.1f}pp"
    elif fvg == 'SIBI':
        edge = f"Bear edge: +{bear_pct - bull_pct:.1f}pp"
    else:
        dominant = 'Bull' if bull_pct > bear_pct else 'Bear'
        edge = f"{dominant} dominant: +{abs(bull_pct - bear_pct):.1f}pp"

    print(f"{label:<16} {t:>6} {bull_pct:>7.1f}% {bear_pct:>7.1f}% {neut_pct:>9.1f}%  {edge}")

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

FVG CONFLUENCE ANALYSIS - RTH SESSION DIRECTION

[1/4] Loading data...
      Loaded 4,974,698 rows
      Filtered to 4,161,727 rows (2013-01-01 to 2025-12-31)

[2/4] Resampling to 5-minute bars...
      Resampled to 854,417 bars

[3/4] Analysing trading days...
      Days processed: 3,130 | Skipped: 653

[4/4] Generating report...

SECTION 1: WINDOW 1 (9:30–10:00am) FPFVG  →  RTH SESSION (9:30am open vs 4:15pm close)

  9:30–10am FPFVG = BULLISH FVG (BISI)
  --------------------------------------------------
  Sample : 1000
  Bullish:  712  ( 71.2%)
  Bearish:  286  ( 28.6%)
  Neutral:    2  (  0.2%)

  9:30–10am FPFVG = BEARISH FVG (SIBI)
  --------------------------------------------------
  Sample : 931
  Bullish:  346  ( 37.2%)
  Bearish:  582  ( 62.5%)
  Neutral:    3  (  0.3%)

  9:30–10am FPFVG = NO FVG
  --------------------------------------------------
  Sample : 1199
  Bullish:  656  ( 54.7%)
  Bearish:  542  ( 45.2%)
  Neutral:    1  (  0.1%)

SECTION 2: WINDOW 2 (10:00–11:

In [4]:
## FVG Confluence Analysis - RTH Session Direction + Violation Tracking
## Tracks FPFVG in 9:30-10am and 10-11am windows
## Measures how the RTH session (9:30am open vs 4:15pm close) resolves
## Violation = price trades through the entire FVG within the same window it formed

import pandas as pd
import numpy as np
from collections import defaultdict

file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"

print("=" * 100)
print("FVG CONFLUENCE ANALYSIS - RTH SESSION DIRECTION + VIOLATION TRACKING")
print("=" * 100)

# ============================================================
# DATA LOADING AND RESAMPLING
# ============================================================

print("\n[1/4] Loading data...")
df = pd.read_csv(file_path)
print(f"      Loaded {len(df):,} rows")

df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.sort_index()

start_date = pd.to_datetime('2013-01-01')
end_date   = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]
print(f"      Filtered to {len(df):,} rows ({start_date.date()} to {end_date.date()})")

print("\n[2/4] Resampling to 5-minute bars...")
df_5min = df.resample('5min').agg({
    'Open':  'first',
    'High':  'max',
    'Low':   'min',
    'Close': 'last'
}).dropna()
print(f"      Resampled to {len(df_5min):,} bars")
df = df_5min

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def detect_fpfvg(bar_data):
    """
    Detect the First Presented Fair Value Gap in a slice of bars.
    BISI (Bullish): candle_3 low > candle_1 high
    SIBI (Bearish): candle_3 high < candle_1 low

    Returns a dict with:
        orientation : 'BISI', 'SIBI', or 'None'
        fvg_top     : top of the FVG zone (nan if no FVG)
        fvg_bot     : bottom of the FVG zone (nan if no FVG)
        formed_idx  : integer iloc of candle_3 in bar_data (nan if no FVG)
    """
    if len(bar_data) < 3:
        return {'orientation': 'None', 'fvg_top': np.nan, 'fvg_bot': np.nan, 'formed_idx': np.nan}

    for i in range(1, len(bar_data) - 1):
        c1 = bar_data.iloc[i - 1]
        c3 = bar_data.iloc[i + 1]

        if c3['Low'] > c1['High']:
            return {'orientation': 'BISI', 'fvg_top': c3['Low'], 'fvg_bot': c1['High'], 'formed_idx': i + 1}

        if c3['High'] < c1['Low']:
            return {'orientation': 'SIBI', 'fvg_top': c1['Low'], 'fvg_bot': c3['High'], 'formed_idx': i + 1}

    return {'orientation': 'None', 'fvg_top': np.nan, 'fvg_bot': np.nan, 'formed_idx': np.nan}


def get_fvg_state(fvg_info, bar_data):
    """
    Classify the FVG outcome within its window into one of three states:
      'intact'            - price never traded through the full FVG
      'violated_reversed' - price broke through but final bar closed back
                            on the correct side (BISI: close > fvg_top,
                            SIBI: close < fvg_bot)
      'violated_failed'   - price broke through and stayed through by
                            window end
    """
    if fvg_info['orientation'] == 'None' or np.isnan(fvg_info['formed_idx']):
        return 'intact'

    after = bar_data.iloc[int(fvg_info['formed_idx']) + 1:]
    if after.empty:
        return 'intact'

    fvg_top     = fvg_info['fvg_top']
    fvg_bot     = fvg_info['fvg_bot']
    final_close = after['Close'].iloc[-1]

    if fvg_info['orientation'] == 'BISI':
        violated = bool((after['Low'] < fvg_bot).any())
        if not violated:
            return 'intact'
        return 'violated_reversed' if final_close > fvg_top else 'violated_failed'
    else:  # SIBI
        violated = bool((after['High'] > fvg_top).any())
        if not violated:
            return 'intact'
        return 'violated_reversed' if final_close < fvg_bot else 'violated_failed' 


def safe_pct(num, denom):
    return (num / denom * 100) if denom > 0 else 0


def make_stat():
    return {'total': 0, 'bullish': 0, 'bearish': 0, 'neutral': 0}


# ============================================================
# TRACKING STRUCTURES
# ============================================================

FVG_STATES = ['BISI_intact', 'BISI_violated_reversed', 'BISI_violated_failed', 'SIBI_intact', 'SIBI_violated_reversed', 'SIBI_violated_failed', 'None']
FVG_VALUES = ['BISI', 'SIBI', 'None']

window1_stats = {s: make_stat() for s in FVG_STATES}
window2_stats = {s: make_stat() for s in FVG_STATES}

combo_stats = {}
for v1 in FVG_VALUES:
    combo_stats[v1] = {}
    for v2 in FVG_VALUES:
        combo_stats[v1][v2] = make_stat()

all_dates = df.index.normalize().unique()
days_processed = 0
days_skipped   = 0

# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

print("\n[3/4] Analysing trading days...")

for date in all_dates:
    date_str = date.strftime('%Y-%m-%d')

    rth_open_time  = pd.Timestamp(f"{date_str} 09:30:00")
    window1_end    = pd.Timestamp(f"{date_str} 10:00:00")
    window2_end    = pd.Timestamp(f"{date_str} 11:00:00")
    rth_close_time = pd.Timestamp(f"{date_str} 16:15:00")

    rth_open_bar = df.loc[rth_open_time:window1_end - pd.Timedelta(minutes=1)]
    if rth_open_bar.empty:
        days_skipped += 1
        continue
    rth_open_price = rth_open_bar['Open'].iloc[0]

    rth_close_bar = df.loc[rth_close_time - pd.Timedelta(minutes=5):rth_close_time]
    if rth_close_bar.empty:
        fallback = df.loc[rth_open_time:rth_close_time]
        if fallback.empty:
            days_skipped += 1
            continue
        rth_close_price = fallback['Close'].iloc[-1]
    else:
        rth_close_price = rth_close_bar['Close'].iloc[-1]

    if rth_close_price > rth_open_price:
        session_dir = 'bullish'
    elif rth_close_price < rth_open_price:
        session_dir = 'bearish'
    else:
        session_dir = 'neutral'

    # Window 1
    w1_data = df.loc[rth_open_time:window1_end - pd.Timedelta(minutes=1)]
    fvg1    = detect_fpfvg(w1_data)
    w1_ori  = fvg1['orientation']
    w1_outcome = get_fvg_state(fvg1, w1_data)
    w1_state   = 'None' if w1_ori == 'None' else f"{w1_ori}_{w1_outcome}"

    # Window 2
    w2_data = df.loc[window1_end:window2_end - pd.Timedelta(minutes=1)]
    fvg2    = detect_fpfvg(w2_data)
    w2_ori  = fvg2['orientation']
    w2_outcome = get_fvg_state(fvg2, w2_data)
    w2_state   = 'None' if w2_ori == 'None' else f"{w2_ori}_{w2_outcome}"

    window1_stats[w1_state]['total']     += 1
    window1_stats[w1_state][session_dir] += 1

    window2_stats[w2_state]['total']     += 1
    window2_stats[w2_state][session_dir] += 1

    combo_stats[w1_ori][w2_ori]['total']     += 1
    combo_stats[w1_ori][w2_ori][session_dir] += 1

    days_processed += 1

print(f"      Days processed: {days_processed:,} | Skipped: {days_skipped:,}")

# ============================================================
# REPORTING
# ============================================================

print("\n[4/4] Generating report...")

STATE_LABELS = {
    'BISI_intact':            'BISI intact            (Bull FVG, held)',
    'BISI_violated_reversed': 'BISI violated+reversed (Bull FVG, swept & recovered)',
    'BISI_violated_failed':   'BISI violated+failed   (Bull FVG, broke down)',
    'SIBI_intact':            'SIBI intact            (Bear FVG, held)',
    'SIBI_violated_reversed': 'SIBI violated+reversed (Bear FVG, swept & recovered)',
    'SIBI_violated_failed':   'SIBI violated+failed   (Bear FVG, broke up)',
    'None':                   'No FVG',
}

def print_outcome(stats, indent="  "):
    t = stats['total']
    if t == 0:
        print(f"{indent}No data")
        return
    print(f"{indent}Sample : {t}")
    print(f"{indent}Bullish: {stats['bullish']:4d}  ({safe_pct(stats['bullish'], t):5.1f}%)")
    print(f"{indent}Bearish: {stats['bearish']:4d}  ({safe_pct(stats['bearish'], t):5.1f}%)")
    print(f"{indent}Neutral: {stats['neutral']:4d}  ({safe_pct(stats['neutral'], t):5.1f}%)")


# --- Section 1: Window 1 violation-aware ---
print("\n" + "=" * 100)
print("SECTION 1: WINDOW 1 (9:30–10:00am) FPFVG  →  RTH SESSION  [violation-aware]")
print("=" * 100)
print("  Intact   = FVG formed and price did NOT fully trade through it before 10am")
print("  Violated = FVG formed but price traded through the entire gap before 10am")

for state in FVG_STATES:
    print(f"\n  9:30–10am = {STATE_LABELS[state]}")
    print(f"  {'-' * 55}")
    print_outcome(window1_stats[state])


# --- Section 2: Window 2 violation-aware ---
print("\n" + "=" * 100)
print("SECTION 2: WINDOW 2 (10:00–11:00am) FPFVG  →  RTH SESSION  [violation-aware]")
print("=" * 100)

for state in FVG_STATES:
    print(f"\n  10am–11am = {STATE_LABELS[state]}")
    print(f"  {'-' * 55}")
    print_outcome(window2_stats[state])


# --- Section 3: Confluence (orientation only) ---
print("\n" + "=" * 100)
print("SECTION 3: CONFLUENCE — WINDOW 1 + WINDOW 2 FPFVG  →  RTH SESSION")
print("=" * 100)
print("\nAll 9 combinations (orientation only — intact + violated combined):\n")

for v1 in FVG_VALUES:
    for v2 in FVG_VALUES:
        l1 = {'BISI': 'BISI (Bull)', 'SIBI': 'SIBI (Bear)', 'None': 'No FVG'}[v1]
        l2 = {'BISI': 'BISI (Bull)', 'SIBI': 'SIBI (Bear)', 'None': 'No FVG'}[v2]
        print(f"  9:30–10am = {l1:<14}  |  10am–11am = {l2:<14}")
        print(f"  {'-' * 60}")
        print_outcome(combo_stats[v1][v2])
        print()


# --- Section 4: Violation impact summary ---
print("=" * 100)
print("SECTION 4: SUMMARY TABLE — VIOLATION IMPACT PER WINDOW")
print("=" * 100)
print("\nKey question: does a violated FVG flip or weaken the directional signal?")

for window_label, wstats in [("WINDOW 1 (9:30–10am)", window1_stats),
                               ("WINDOW 2 (10am–11am)",  window2_stats)]:
    print(f"\n  {window_label}")
    print(f"  {'State':<32} {'N':>6} {'Bull%':>8} {'Bear%':>8} {'Edge':>25}")
    print(f"  {'-' * 83}")
    for state in FVG_STATES:
        s = wstats[state]
        t = s['total']
        bull_pct = safe_pct(s['bullish'], t)
        bear_pct = safe_pct(s['bearish'], t)
        if 'BISI' in state:
            edge = f"Bull edge: {bull_pct - bear_pct:+.1f}pp"
        elif 'SIBI' in state:
            edge = f"Bear edge: {bear_pct - bull_pct:+.1f}pp"
        else:
            dom = 'Bull' if bull_pct >= bear_pct else 'Bear'
            edge = f"{dom} dom: +{abs(bull_pct - bear_pct):.1f}pp"
        print(f"  {STATE_LABELS[state]:<32} {t:>6} {bull_pct:>7.1f}% {bear_pct:>7.1f}% {edge:>25}")


# --- Section 5: Confluence summary table ---
print("\n\n" + "=" * 100)
print("SECTION 5: CONFLUENCE SUMMARY TABLE")
print("=" * 100)
print(f"\n{'Window 1 (9:30-10)':<20} {'Window 2 (10-11)':<20} {'N':>6} {'Bull%':>8} {'Bear%':>8} {'Neutral%':>10}")
print("-" * 76)

for v1 in FVG_VALUES:
    for v2 in FVG_VALUES:
        l1 = {'BISI': 'BISI', 'SIBI': 'SIBI', 'None': 'No FVG'}[v1]
        l2 = {'BISI': 'BISI', 'SIBI': 'SIBI', 'None': 'No FVG'}[v2]
        s = combo_stats[v1][v2]
        t = s['total']
        print(f"{l1:<20} {l2:<20} {t:>6} "
              f"{safe_pct(s['bullish'], t):>7.1f}% "
              f"{safe_pct(s['bearish'], t):>7.1f}% "
              f"{safe_pct(s['neutral'], t):>9.1f}%")


# --- Section 6: W1 isolated ---
print("\n\n" + "=" * 100)
print("SECTION 6: WINDOW 1 (9:30–10:00am) — ISOLATED TABLE WITH VIOLATION")
print("=" * 100)
print(f"\n{'9:30-10am FVG':<32} {'N':>6} {'Bull%':>8} {'Bear%':>8} {'Neutral%':>10}  {'Edge'}")
print("-" * 90)

for state in FVG_STATES:
    s = window1_stats[state]
    t = s['total']
    bull_pct = safe_pct(s['bullish'], t)
    bear_pct = safe_pct(s['bearish'], t)
    neut_pct = safe_pct(s['neutral'], t)
    if 'BISI' in state:
        edge = f"Bull edge: {bull_pct - bear_pct:+.1f}pp"
    elif 'SIBI' in state:
        edge = f"Bear edge: {bear_pct - bull_pct:+.1f}pp"
    else:
        dom = 'Bull' if bull_pct > bear_pct else 'Bear'
        edge = f"{dom} dom: +{abs(bull_pct - bear_pct):.1f}pp"
    print(f"{STATE_LABELS[state]:<32} {t:>6} {bull_pct:>7.1f}% {bear_pct:>7.1f}% {neut_pct:>9.1f}%  {edge}")

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

FVG CONFLUENCE ANALYSIS - RTH SESSION DIRECTION + VIOLATION TRACKING

[1/4] Loading data...
      Loaded 4,974,698 rows
      Filtered to 4,161,727 rows (2013-01-01 to 2025-12-31)

[2/4] Resampling to 5-minute bars...
      Resampled to 854,417 bars

[3/4] Analysing trading days...
      Days processed: 3,130 | Skipped: 653

[4/4] Generating report...

SECTION 1: WINDOW 1 (9:30–10:00am) FPFVG  →  RTH SESSION  [violation-aware]
  Intact   = FVG formed and price did NOT fully trade through it before 10am
  Violated = FVG formed but price traded through the entire gap before 10am

  9:30–10am = BISI intact            (Bull FVG, held)
  -------------------------------------------------------
  Sample : 743
  Bullish:  551  ( 74.2%)
  Bearish:  190  ( 25.6%)
  Neutral:    2  (  0.3%)

  9:30–10am = BISI violated+reversed (Bull FVG, swept & recovered)
  -------------------------------------------------------
  Sample : 96
  Bullish:   67  ( 69.8%)
  Bearish:   29  ( 30.2%)
  Neutral:    0  (